# Road Anomaly Detection - Colab Pro+ Baseline

This notebook prepares public road-damage datasets, trains a YOLO baseline, saves results to Google Drive, and runs optional hard-negative validation.

Recommended first run: keep `RUN_MODE = "fast"`. Use `RUN_MODE = "full"` only after the first baseline is working.

In [ ]:
# Runtime check
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

In [ ]:
# Configuration
RUN_MODE = "fast"  #@param ["fast", "full"]
BASE_MODEL = "yolov8n.pt"  #@param ["yolov8n.pt", "yolov8s.pt"]
EPOCHS = 20  #@param {type:"integer"}
IMGSZ = 640  #@param {type:"integer"}
BATCH = 16  #@param {type:"integer"}
FORCE_REBUILD_DATA = False  #@param {type:"boolean"}
CACHE_DATA_TO_DRIVE = True  #@param {type:"boolean"}

REPO_URL = "https://github.com/Mythi46/road-anomaly-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/road-anomaly-detection"
WORK_ROOT = "/content/road-anomaly-detection"
DATA_ROOT = "/content/road_data/public"
RUN_NAME = f"{RUN_MODE}_{BASE_MODEL.replace('.pt','')}_e{EPOCHS}_img{IMGSZ}"

In [ ]:
# Mount Drive, clone repo, install Colab-safe dependencies
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json, time, signal

drive.mount('/content/drive')
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_ROOT}/runs").mkdir(parents=True, exist_ok=True)
Path(f"{DRIVE_ROOT}/datasets").mkdir(parents=True, exist_ok=True)

if Path(WORK_ROOT).exists():
    shutil.rmtree(WORK_ROOT)
subprocess.run(["git", "clone", REPO_URL, WORK_ROOT], check=True)
os.chdir(WORK_ROOT)
# Colab Python 3.12 can hit ABI conflicts after mixed package upgrades.
# Install binary-sensitive packages, restart once, then import-test on the next run.
deps_marker = Path("/content/.road_anomaly_colab_deps_ready")
if not deps_marker.exists():
    colab_packages = [
        "numpy==1.26.4",
        "pillow==10.4.0",
        "PyYAML>=6.0",
        "opencv-python-headless==4.10.0.84",
        "ultralytics==8.3.228",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--force-reinstall", *colab_packages], check=True)
    deps_marker.write_text("ok", encoding="utf-8")
    print("Dependencies installed. Restarting Colab runtime once; after reconnect, run this cell again.")
    os.kill(os.getpid(), signal.SIGKILL)

# Smoke test the binary imports after the restart, before expensive data work.
import numpy as np
import PIL
from ultralytics import YOLO
print("numpy:", np.__version__)
print("pillow:", PIL.__version__)
print("ultralytics import: ok")
print("repo ready:", WORK_ROOT)

In [ ]:
# Helper functions
from pathlib import Path
import subprocess, sys, shutil, os, json, time

REPO = Path(WORK_ROOT)
DATA = Path(DATA_ROOT)
RAW = DATA / "raw"
EXTRACTED = DATA / "extracted"
CONVERTED = DATA / "converted"
CACHE_ZIP = Path(DRIVE_ROOT) / "datasets" / f"public_converted_{RUN_MODE}_v1.zip"

FAST_DATASETS = [
    "n_rdd2024",
    "mwpd",
    "road_damage_potholes_cracks_manholes",
    "water_filled_dry_potholes",
    "attain",
]
FULL_DATASETS = ["rdd2022", "rdd2020", *FAST_DATASETS]


def run(args, cwd=REPO):
    print("RUN:", " ".join(str(a) for a in args))
    subprocess.run([str(a) for a in args], cwd=str(cwd), check=True)


def prepare(dataset_id):
    run([sys.executable, "scripts/prepare_public_datasets.py", "--data-root", DATA, "--dataset", dataset_id, "--include-large", "--max-file-size-mb", "5"])


def extract_archive(archive, output_root):
    run([sys.executable, "scripts/extract_archive.py", "--archive", archive, "--output-root", output_root])


def remap(input_root, output_root, target_mode="multi-bucket", data_yaml=None, source_names=None):
    args = [sys.executable, "scripts/remap_yolo_dataset.py", "--input-root", input_root, "--output-root", output_root, "--target-mode", target_mode, "--image-mode", "hardlink", "--unknown-policy", "skip"]
    if data_yaml:
        args += ["--data-yaml", data_yaml]
    for name in source_names or []:
        args += ["--source-name", name]
    run(args)


def convert_voc(input_root, image_root, output_root, class_mode="v0"):
    run([sys.executable, "scripts/convert_rdd_xml_to_yolo.py", "--input-root", input_root, "--image-root", image_root, "--output-root", output_root, "--class-mode", class_mode, "--split", "train", "--copy-images", "--write-empty-labels"])


def find_one(root, pattern):
    matches = sorted(Path(root).rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"No match for {pattern} under {root}")
    return matches[0]

In [ ]:
# Prepare public datasets, or restore converted cache from Drive
if CACHE_ZIP.exists() and not FORCE_REBUILD_DATA:
    print("Restoring converted dataset cache:", CACHE_ZIP)
    if DATA.exists():
        shutil.rmtree(DATA)
    DATA.parent.mkdir(parents=True, exist_ok=True)
    run(["unzip", "-q", str(CACHE_ZIP), "-d", str(DATA.parent)], cwd=Path("/content"))
else:
    DATA.mkdir(parents=True, exist_ok=True)
    selected = FULL_DATASETS if RUN_MODE == "full" else FAST_DATASETS
    print("Preparing datasets:", selected)

    if "n_rdd2024" in selected:
        prepare("n_rdd2024")
        run([sys.executable, "scripts/extract_nested_zips.py", "--zip", RAW/"n_rdd2024"/"n_rdd2024.zip", "--output-root", EXTRACTED/"n_rdd2024", "--member-pattern", "*_txt.zip"])
        remap(EXTRACTED/"n_rdd2024", CONVERTED/"n_rdd2024_multi", "multi-bucket")

    if "mwpd" in selected:
        prepare("mwpd")
        extract_archive(RAW/"mwpd"/"mwpd.zip", EXTRACTED/"mwpd")
        mwpd_root = find_one(EXTRACTED/"mwpd", "data.yaml").parent
        remap(mwpd_root, CONVERTED/"mwpd_v0", "v0")

    if "road_damage_potholes_cracks_manholes" in selected:
        prepare("road_damage_potholes_cracks_manholes")
        extract_archive(RAW/"road_damage_potholes_cracks_manholes"/"archive.zip", EXTRACTED/"road_damage_potholes_cracks_manholes")
        rd_root = EXTRACTED/"road_damage_potholes_cracks_manholes"/"data"
        remap(rd_root, CONVERTED/"road_damage_potholes_cracks_manholes_multi", "multi-bucket", source_names=["pothole", "crack", "maintenance_hole"])

    if "water_filled_dry_potholes" in selected:
        prepare("water_filled_dry_potholes")
        extract_archive(RAW/"water_filled_dry_potholes"/"water_filled_dry_potholes.zip", EXTRACTED/"water_filled_dry_potholes")
        water_root = find_one(EXTRACTED/"water_filled_dry_potholes", "ReadMe.txt").parent
        convert_voc(water_root/"XML", water_root/"IMG", CONVERTED/"water_filled_dry_potholes_v0", "v0")

    if "attain" in selected:
        prepare("attain")
        extract_archive(RAW/"attain"/"attain.zip", EXTRACTED/"attain")
        attain_base = EXTRACTED/"attain"/"Attain"/"Attain"
        os_v1 = attain_base/"Attain_SMP_OS_V1.0"
        ws_v1 = attain_base/"Attain_SMP_WS_V1.0"
        ws_v2 = attain_base/"Attain_SMP_WS_V2.0"
        remap(os_v1, CONVERTED/"attain_os_v1_multi", "multi-bucket", data_yaml=os_v1/"Attain_SMP_OS_V1.0_data.yaml")
        remap(ws_v1, CONVERTED/"attain_ws_v1_multi", "multi-bucket", data_yaml=ws_v1/"Attain_SMP_WS_V1.0_data.yaml")
        convert_voc(ws_v2/"Labels", ws_v2/"Images", CONVERTED/"attain_ws_v2_multi", "multi-bucket")

    if RUN_MODE == "full":
        prepare("rdd2022")
        run([sys.executable, "scripts/extract_rdd2022.py", "--zip", RAW/"rdd2022"/"RDD2022_released_through_CRDDC2022.zip", "--output-root", EXTRACTED/"rdd2022"])
        convert_voc(EXTRACTED/"rdd2022", EXTRACTED/"rdd2022", CONVERTED/"rdd2022_v0", "v0")

        prepare("rdd2020")
        extract_archive(RAW/"rdd2020"/"rdd2020.zip", EXTRACTED/"rdd2020_outer")
        extract_archive(EXTRACTED/"rdd2020_outer"/"train.tar.gz", EXTRACTED/"rdd2020_train")
        convert_voc(EXTRACTED/"rdd2020_train", EXTRACTED/"rdd2020_train", CONVERTED/"rdd2020_v0", "v0")

    if CACHE_DATA_TO_DRIVE:
        print("Writing converted dataset cache to Drive:", CACHE_ZIP)
        CACHE_ZIP.parent.mkdir(parents=True, exist_ok=True)
        if CACHE_ZIP.exists():
            CACHE_ZIP.unlink()
        run(["zip", "-qr", str(CACHE_ZIP), "public/converted"], cwd=DATA.parent)

print("Converted datasets:")
for path in sorted(CONVERTED.glob("*")):
    print(" -", path)

In [ ]:
# Build combined YOLO data.yaml
combined_yaml = Path(DRIVE_ROOT) / "datasets" / f"combined_{RUN_MODE}_v1.yaml"
source_args = []
if RUN_MODE == "fast":
    for source in [
        "n_rdd2024_multi",
        "mwpd_v0",
        "road_damage_potholes_cracks_manholes_multi",
        "water_filled_dry_potholes_v0",
        "attain_os_v1_multi",
        "attain_ws_v1_multi",
        "attain_ws_v2_multi",
    ]:
        source_args += ["--source", source]
run([sys.executable, "scripts/write_combined_yolo_yaml.py", "--converted-root", CONVERTED, "--output", combined_yaml, *source_args])
print(combined_yaml.read_text())

In [ ]:
# Train YOLO baseline
from ultralytics import YOLO

runs_dir = Path(DRIVE_ROOT) / "runs"
runs_dir.mkdir(parents=True, exist_ok=True)

model = YOLO(BASE_MODEL)
results = model.train(
    data=str(combined_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=str(runs_dir),
    name=RUN_NAME,
    exist_ok=True,
    patience=5,
    cache=False,
    plots=True,
)
print("training run:", runs_dir / RUN_NAME)

In [ ]:
# Validate and save key paths
from ultralytics import YOLO

run_dir = Path(DRIVE_ROOT) / "runs" / RUN_NAME
best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"
print("best:", best_pt, best_pt.exists())
print("last:", last_pt, last_pt.exists())

model = YOLO(str(best_pt if best_pt.exists() else last_pt))
metrics = model.val(data=str(combined_yaml), imgsz=IMGSZ, batch=BATCH, device=0, project=str(run_dir), name="val", exist_ok=True, plots=True)
print(metrics)

In [ ]:
# Optional hard-negative / private validation from Drive
# Put private evaluation images here, for example repaired road, manholes, lane markings, shadows:
#   MyDrive/road-anomaly-detection/private_eval/hard_negative/

hard_negative_dir = Path(DRIVE_ROOT) / "private_eval" / "hard_negative"
if hard_negative_dir.exists():
    print("Running hard-negative prediction on", hard_negative_dir)
    model.predict(
        source=str(hard_negative_dir),
        imgsz=IMGSZ,
        conf=0.15,
        device=0,
        save=True,
        save_txt=True,
        save_conf=True,
        project=str(run_dir),
        name="hard_negative_predictions",
        exist_ok=True,
    )
else:
    print("No hard-negative folder found yet:", hard_negative_dir)

In [ ]:
# Simple inference-speed check on validation images
import glob, time
from statistics import mean

val_dirs = []
summary_path = combined_yaml.with_suffix(".summary.json")
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    val_dirs = summary.get("val_dirs", [])

sample_images = []
for val_dir in val_dirs:
    sample_images.extend(glob.glob(str(Path(val_dir) / "*.jpg")))
sample_images = sample_images[:200]

if sample_images:
    times = []
    for image_path in sample_images:
        t0 = time.perf_counter()
        model.predict(source=image_path, imgsz=IMGSZ, conf=0.25, device=0, verbose=False)
        times.append(time.perf_counter() - t0)
    print("images:", len(times))
    print("avg seconds/image:", mean(times))
    print("approx fps:", 1.0 / mean(times))
else:
    print("No validation images found for speed test.")